# 01 · Why keyword search misses meaning
Level L1 · Part 1 · builds on no earlier lessons

## Goal
By the end of this lesson you can show a recipe query that keyword matching gets wrong,
and explain why searching by *meaning* finds the right recipe.

## The idea
Most search boxes match **words**: a recipe is returned only if it contains the words you typed.
People do not search that way. They describe what they *want* ("something warm for a cold day"),
and the right recipe may use completely different words ("serve hot").
**Semantic search** compares meanings instead of spellings.
Analogy: a librarian who has read every book can hand you the right one even if you
forget the title; a librarian who only checks the title page cannot.
The rest of this course is about how to turn meaning into numbers and search those numbers fast.

## Picture

```mermaid
flowchart LR
    Q["query: something warm for a cold day"] --> K["keyword search: compare words"]
    Q --> S["semantic search: compare meanings"]
    K --> K1["no recipe shares a word -> nothing"]
    S --> S1["Tomato soup is hot and comforting -> found"]
```

## Step by step

### 1. Five recipes
Our running example is a **recipe finder**. It starts tiny: five recipes, each with a name and a
one-line description. We will grow this list in later lessons.

In [1]:
recipes = {
    "Tomato soup": "Simmer tomatoes, onion and garlic, blend until smooth and serve hot with crusty bread.",
    "Mango salsa": "Diced mango, red onion, lime juice and chili. Fresh and zesty, great with tacos.",
    "Chocolate brownies": "Rich fudgy squares of chocolate baked in a tray.",
    "Caesar salad": "Crisp romaine lettuce, croutons, parmesan and a creamy dressing.",
    "Lemon sorbet": "Frozen lemon dessert, sharp and refreshing on summer afternoons.",
}
for name, text in recipes.items():
    print(f"{name:20} {text}")

Tomato soup          Simmer tomatoes, onion and garlic, blend until smooth and serve hot with crusty bread.
Mango salsa          Diced mango, red onion, lime juice and chili. Fresh and zesty, great with tacos.
Chocolate brownies   Rich fudgy squares of chocolate baked in a tray.
Caesar salad         Crisp romaine lettuce, croutons, parmesan and a creamy dressing.
Lemon sorbet         Frozen lemon dessert, sharp and refreshing on summer afternoons.


### 2. Turn text into a set of words
Keyword search first breaks text into **tokens** (here: lowercase words without punctuation).
It also drops **stop words** such as "a" and "for", which appear everywhere and carry little meaning.

In [2]:
import re

STOP_WORDS = {"a", "an", "and", "the", "for", "of", "on", "in", "with", "until", "great"}

def tokenize(text):
    words = re.findall(r"[a-z]+", text.lower())
    return {w for w in words if w not in STOP_WORDS}

print(sorted(tokenize("Something warm for a cold day")))
print(sorted(tokenize(recipes["Tomato soup"])))

['cold', 'day', 'something', 'warm']
['blend', 'bread', 'crusty', 'garlic', 'hot', 'onion', 'serve', 'simmer', 'smooth', 'tomatoes']


### 3. Keyword search: count shared words
The simplest keyword search scores each recipe by how many query words it contains,
and keeps recipes with a score above zero. Real engines (for example BM25, lesson 19)
weigh words more cleverly, but they still need the words to match.

In [3]:
def keyword_search(query, recipes):
    q = tokenize(query)
    scores = {name: len(q & tokenize(text)) for name, text in recipes.items()}
    hits = {name: s for name, s in scores.items() if s > 0}
    return sorted(hits.items(), key=lambda item: -item[1])

print("hot tomato soup             ->", keyword_search("hot tomato soup", recipes))
print("something warm for a cold day ->", keyword_search("something warm for a cold day", recipes))

hot tomato soup             -> [('Tomato soup', 1)]
something warm for a cold day -> []


Keyword search works when you already know the recipe's words ("hot tomato soup"), though only
one word, "hot", matched: "tomato" is not "tomatoes", and the description never says "soup".
For the question a real person asks, it returns **an empty list**.
The problem is not the scoring; the words simply never overlap:

In [4]:
query_words = tokenize("something warm for a cold day")
soup_words = tokenize(recipes["Tomato soup"])
print("query words:      ", sorted(query_words))
print("soup words:       ", sorted(soup_words))
print("shared words:     ", sorted(query_words & soup_words))

query words:       ['cold', 'day', 'something', 'warm']
soup words:        ['blend', 'bread', 'crusty', 'garlic', 'hot', 'onion', 'serve', 'simmer', 'smooth', 'tomatoes']
shared words:      []


### 4. Patch it with synonyms?
A common first fix is a hand-written synonym list: "warm" also means "hot".
This rescues *this* query, but somebody has to write every pair in advance,
for every word, cuisine and language.

In [5]:
SYNONYMS = {"warm": {"hot"}, "cold": {"chilly", "winter"}}

def keyword_search_with_synonyms(query, recipes):
    q = tokenize(query)
    for word in list(q):
        q |= SYNONYMS.get(word, set())
    scores = {name: len(q & tokenize(text)) for name, text in recipes.items()}
    return sorted(((n, s) for n, s in scores.items() if s > 0), key=lambda item: -item[1])

print(keyword_search_with_synonyms("something warm for a cold day", recipes))
print(keyword_search_with_synonyms("comforting dish for winter", recipes))

[('Tomato soup', 1)]
[]


The synonym list found the soup for the first query, but a new phrasing ("comforting dish for
winter") fails again, because nobody wrote "comforting -> hot". Synonym lists never end.

### 5. Search by meaning instead
Imagine a helper read each recipe and wrote down what it *means* as two scores from 0 to 1:
how **warm** the dish is when served, and how **comforting** (hearty) it feels.
Words disappear; only meaning is left. (In lesson 05 a model writes these numbers for us.)

In [6]:
meaning = {               # (warm, comforting), scored by hand
    "Tomato soup":        (0.9, 0.9),
    "Mango salsa":        (0.1, 0.2),
    "Chocolate brownies": (0.4, 0.8),
    "Caesar salad":       (0.1, 0.3),
    "Lemon sorbet":       (0.0, 0.2),
}
query_meaning = (1.0, 1.0)  # "something warm for a cold day": very warm, very comforting
print("query:", query_meaning)
for name, scores in meaning.items():
    print(f"{name:20} warm={scores[0]:.1f} comforting={scores[1]:.1f}")

query: (1.0, 1.0)
Tomato soup          warm=0.9 comforting=0.9
Mango salsa          warm=0.1 comforting=0.2
Chocolate brownies   warm=0.4 comforting=0.8
Caesar salad         warm=0.1 comforting=0.3
Lemon sorbet         warm=0.0 comforting=0.2


To compare the query with a recipe, multiply the matching scores and add them up.
A recipe that is high on the same things as the query gets a high total.
(Lesson 03 names this calculation and shows other ways to measure closeness.)

In [7]:
def meaning_search(query_scores, meaning):
    totals = {}
    for name, scores in meaning.items():
        totals[name] = sum(q * r for q, r in zip(query_scores, scores, strict=True))
    return sorted(totals.items(), key=lambda item: -item[1])

for name, total in meaning_search(query_meaning, meaning):
    print(f"{name:20} {total:.2f}")

Tomato soup          1.80
Chocolate brownies   1.20
Caesar salad         0.40
Mango salsa          0.30
Lemon sorbet         0.20


**Tomato soup** comes first, with brownies (comforting, but not served hot) second and the
cold sorbet last. Every recipe gets a score, so we get a *ranking*, not just "match / no match".

### 6. Side by side
The same question, both ways:

In [8]:
question = "something warm for a cold day"
kw = keyword_search(question, recipes)
sem = meaning_search(query_meaning, meaning)
print(f"question:         {question!r}")
print(f"keyword search:   {[name for name, _ in kw] or 'no results'}")
print(f"meaning search:   top-3 = {[name for name, _ in sem[:3]]}")

question:         'something warm for a cold day'
keyword search:   no results
meaning search:   top-3 = ['Tomato soup', 'Chocolate brownies', 'Caesar salad']


## What just happened
- We tokenized text into word sets (step 2) and scored recipes by shared words (step 3).
- Keyword search found "Tomato soup" for "hot tomato soup", but returned **no results** for
  "something warm for a cold day", because the query and the soup share **zero** words.
- A synonym list fixed one query and failed on the next ("comforting dish for winter").
- Describing each recipe by *meaning* scores (warm, comforting) put Tomato soup first with 1.80,
  and ranked every recipe instead of giving an all-or-nothing answer.
- The open questions for the next lessons: what exactly are these lists of numbers (02),
  how do we measure closeness (03), and who writes the numbers for thousands of recipes (05)?

## Common mistakes

**1. Comparing raw text without normalising it.** Splitting on spaces keeps capital letters and
punctuation, so "Hot" never matches "hot" and "bread." never matches "bread". The check below fails:

In [9]:
try:
    def naive_tokenize(text):
        return set(text.split())

    shared = naive_tokenize("Hot soup") & naive_tokenize(recipes["Tomato soup"])
    print("shared words:", sorted(shared))
    assert "hot" in shared or "Hot" in shared, "keyword match missed 'hot' because 'Hot' has a capital H"
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

shared words: []
AssertionError: keyword match missed 'hot' because 'Hot' has a capital H


Fix: lowercase and strip punctuation before comparing, as `tokenize` does.

In [10]:
print("shared words:", sorted(tokenize("Hot soup") & tokenize(recipes["Tomato soup"])))

shared words: ['hot']


**2. Scoring meanings that do not line up.** Meaning scores only compare if every item uses the
same scores in the same order. Here the query adds a third score ("spicy") the recipes do not have.
`zip(..., strict=True)` turns this silent bug into an error:

In [11]:
try:
    query_with_spice = (1.0, 1.0, 0.0)   # (warm, comforting, spicy)
    meaning_search(query_with_spice, meaning)
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

ValueError: zip() argument 2 is shorter than argument 1


Fix: describe the query and every recipe with the **same** list of scores, in the same order.
Without `strict=True`, `zip` would quietly drop the extra score and you would never notice.

In [12]:
print(meaning_search((1.0, 1.0), meaning)[0])

('Tomato soup', 1.8)


## Try it
1. Add a sixth recipe, `"Beef stew"`, with a description that does not contain "warm" or "cold",
   and your own (warm, comforting) scores. Run both searches for "something warm for a cold day".
   Where does the stew rank?
2. The query "a refreshing treat for a summer afternoon" needs a meaning our two scores cannot
   express. Add a third score, **refreshing**, to every recipe and to the query (same order
   everywhere!), then check that Lemon sorbet comes first.

In [13]:
# your code here

## Key terms
- **keyword search** — finding items that contain the same words as the query (also called lexical search).
- **token** — one unit of text after splitting and cleaning, here a lowercase word.
- **stop word** — a very common word ("a", "for", "the") that keyword search ignores.
- **semantic search** — finding items whose *meaning* is close to the query's, even with no shared words.
- **ranking** — ordering every item by a score, instead of a yes/no match.